# NASA FIRMS fire alerts for an AOI — recent or historical periods

This notebook downloads NASA FIRMS active-fire detections for a user-defined Area of Interest (AOI).

It supports **two date modes**:

- `recent` — download the last *N* days.
- `range` — download an explicit historical period, for example two weeks one year ago.

For historical periods, the notebook first checks the FIRMS **data availability API** and automatically chooses the available Standard Processing (`_SP`) or Near Real-Time (`_NRT`) product for each sensor/date.

The final output is one point dataset containing the original FIRMS attributes plus:

- `sensor_name`
- `firms_source`
- `processing_type`
- `acq_datetime`

The result is saved as a GeoPackage suitable for QGIS and CDSE workflows.

### Project storage convention

Outputs are stored under:

```text
mystorage/
└── FIRMS/
    └── <PROJECT_NAME>/
        └── outputs/
```

The GeoPackage filename follows:

```text
Fire_Alerts_Firms_<PROJECT_NAME>_<start-date>_to_<end-date>.gpkg
```

For example:

```text
Fire_Alerts_Firms_ALB-WF001_2026-08-16_to_2026-08-23.gpkg
```


## 1. Install packages

Run this only if the imports in the next cell fail.

CDSE environments often already contain some or all of these packages.

In [ ]:
%pip install -q geopandas pandas requests pyogrio shapely

## 2. Imports

In [ ]:
import os
from getpass import getpass
import io
from pathlib import Path
from datetime import date, datetime, timedelta

import pandas as pd
import geopandas as gpd
import requests

print("Imports successful.")

## 3. User settings

### Date modes

For the last 14 days:

```python
DATE_MODE = "recent"
DAYS = 14
```

For a specific historical period:

```python
DATE_MODE = "range"
START_DATE = "2025-08-20"
END_DATE   = "2025-09-02"
```

The start and end dates are **inclusive**.

### Sensor selection

The notebook works with logical sensor names rather than forcing a particular FIRMS processing stream.

For older dates it will prefer Standard Processing where available and otherwise use NRT if FIRMS reports that the requested date is available there.

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================

MAP_KEY = os.environ.get("FIRMS_MAP_KEY") or getpass("FIRMS MAP key: ")
if not MAP_KEY.strip():
    raise ValueError("Provide a FIRMS MAP key or set FIRMS_MAP_KEY before starting Jupyter.")

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------

repo_root = Path.home() / "mystorage" / "fire-school"

candidates = [
    repo_root / "data" / "aoi" / "ALB_WF001_AOI.shp",
    Path.cwd() / "data" / "aoi" / "ALB_WF001_AOI.shp",
    Path.cwd().parent / "data" / "aoi" / "ALB_WF001_AOI.shp",
]

AOI_FILE = next((p for p in candidates if p.exists()), None)

if AOI_FILE is None:
    raise FileNotFoundError(
        "Canonical AOI file data/aoi/ALB_WF001_AOI.shp was not found. "
        "Run git pull in the fire-school repository."
    )

print("Using:", AOI_FILE)

ROOT_FOLDER = AOI_FILE.parents[2]

PROJECT_NAME = "ALB-WF001"

MYSTORAGE = ROOT_FOLDER / ""
FIRMS_ROOT = MYSTORAGE / "FIRMS"
PROJECT_DIR = FIRMS_ROOT / PROJECT_NAME
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FIRMS project:", PROJECT_DIR)
print("Output folder:", OUTPUT_DIR)

# ------------------------------------------------------------
# DATE MODE
# ------------------------------------------------------------

# Choose either:
#   "recent" -> last DAYS days
#   "range"  -> explicit START_DATE and END_DATE
DATE_MODE = "range"
#DATE_MODE = "recent"

# Used only when DATE_MODE == "recent"
DAYS = 12

# Used only when DATE_MODE == "range"
# Example: a 14-day period about one year ago
START_DATE = "2026-08-13"
END_DATE   = "2026-08-24"

# ------------------------------------------------------------
# SENSORS
# ------------------------------------------------------------

SELECTED_SENSORS = [
    "VIIRS NOAA-21",
    "VIIRS NOAA-20",
    "VIIRS S-NPP",
    "MODIS",
]

# OUTPUT_FILE is created automatically after the requested
# start/end dates have been resolved.
OUTPUT_LAYER = "fire_alerts"

BASE_AREA_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
BASE_AVAILABILITY_URL = (
    "https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv"
)

# Candidate FIRMS products, in preferred order.
# Standard Processing is preferred for older data when available.
SENSOR_PRODUCTS = {
    "VIIRS NOAA-21": [
        "VIIRS_NOAA21_NRT",
    ],
    "VIIRS NOAA-20": [
        "VIIRS_NOAA20_SP",
        "VIIRS_NOAA20_NRT",
    ],
    "VIIRS S-NPP": [
        "VIIRS_SNPP_SP",
        "VIIRS_SNPP_NRT",
    ],
    "MODIS": [
        "MODIS_SP",
        "MODIS_NRT",
    ],
}

## 4. Resolve the requested date period

This cell converts either the recent-period settings or explicit historical dates into `start_date` and `end_date`.

In [ ]:
if DATE_MODE == "recent":

    if DAYS < 1:
        raise ValueError("DAYS must be at least 1.")

    end_date = date.today()
    start_date = end_date - timedelta(days=DAYS - 1)

elif DATE_MODE == "range":

    start_date = datetime.strptime(START_DATE, "%Y-%m-%d").date()
    end_date = datetime.strptime(END_DATE, "%Y-%m-%d").date()

    if end_date < start_date:
        raise ValueError("END_DATE must be on or after START_DATE.")

else:
    raise ValueError('DATE_MODE must be either "recent" or "range".')

requested_days = (end_date - start_date).days + 1

DATE_RANGE_TAG = (
    f"{start_date:%Y-%m-%d}_to_{end_date:%Y-%m-%d}"
)

OUTPUT_FILE = (
    OUTPUT_DIR
    / f"Fire_Alerts_Firms_{PROJECT_NAME}_{DATE_RANGE_TAG}.gpkg"
)

print("Date mode: ", DATE_MODE)
print("Start date:", start_date)
print("End date:  ", end_date)
print("Days:      ", requested_days)
print("Output:    ", OUTPUT_FILE)

## 5. Read the AOI

FIRMS Area API requests use a bounding box.

The notebook therefore:

1. reads your AOI;
2. converts it to WGS84 (`EPSG:4326`);
3. calculates its bounding box;
4. later clips downloaded points back to the true AOI polygon.

In [ ]:
aoi = gpd.read_file(AOI_FILE)

if aoi.empty:
    raise ValueError("The AOI file contains no features.")

aoi = aoi.to_crs("EPSG:4326")
aoi_geom = aoi.geometry.union_all()

minx, miny, maxx, maxy = aoi.total_bounds
bbox = f"{minx},{miny},{maxx},{maxy}"

print("AOI:", AOI_FILE)
print("CRS:", aoi.crs)
print("FIRMS bounding box:", bbox)

aoi

## 6. Ask FIRMS which product dates are available

This is the main improvement for historical work.

FIRMS maintains separate processing streams such as:

- `VIIRS_NOAA20_SP`
- `VIIRS_NOAA20_NRT`

Their exact date coverage changes over time.

Instead of guessing, this notebook queries FIRMS and uses the returned `min_date` and `max_date` values.

In [ ]:
availability_url = f"{BASE_AVAILABILITY_URL}/{MAP_KEY}/all"

availability = pd.read_csv(availability_url)

availability["min_date"] = pd.to_datetime(
    availability["min_date"]
).dt.date

availability["max_date"] = pd.to_datetime(
    availability["max_date"]
).dt.date

availability

## 7. Assign each requested day to an available FIRMS product

For each selected sensor, the notebook checks every requested date.

Where Standard Processing is available it is preferred. If not, an available NRT product is used.

The output of this step is a list of contiguous date segments such as:

```text
VIIRS NOAA-20 | VIIRS_NOAA20_SP  | 2025-08-20 → 2025-09-02
```

If FIRMS reports no available product for a requested day, the notebook prints a warning.

In [ ]:
availability_lookup = {
    row["data_id"]: (row["min_date"], row["max_date"])
    for _, row in availability.iterrows()
}

download_segments = []

for sensor_name in SELECTED_SENSORS:

    if sensor_name not in SENSOR_PRODUCTS:
        raise ValueError(f"Unknown sensor: {sensor_name}")

    candidate_products = SENSOR_PRODUCTS[sensor_name]

    daily_assignments = []

    current_day = start_date

    while current_day <= end_date:

        selected_source = None

        for source in candidate_products:

            if source not in availability_lookup:
                continue

            source_min, source_max = availability_lookup[source]

            if source_min <= current_day <= source_max:
                selected_source = source
                break

        daily_assignments.append((current_day, selected_source))
        current_day += timedelta(days=1)

    # Group consecutive dates using the same source.
    segment_start = None
    segment_source = None
    previous_day = None

    for day_value, source in daily_assignments:

        if source != segment_source:

            if segment_source is not None:
                download_segments.append(
                    {
                        "sensor_name": sensor_name,
                        "source": segment_source,
                        "start": segment_start,
                        "end": previous_day,
                    }
                )

            segment_start = day_value if source is not None else None
            segment_source = source

        previous_day = day_value

    if segment_source is not None:
        download_segments.append(
            {
                "sensor_name": sensor_name,
                "source": segment_source,
                "start": segment_start,
                "end": previous_day,
            }
        )

    missing_days = [
        day_value
        for day_value, source in daily_assignments
        if source is None
    ]

    if missing_days:
        print(
            f"WARNING: {sensor_name} has no reported FIRMS product "
            f"for {len(missing_days)} requested day(s)."
        )

segments_df = pd.DataFrame(download_segments)

if segments_df.empty:
    raise RuntimeError(
        "None of the selected sensors have FIRMS data available "
        "for the requested date range."
    )

segments_df

## 8. Download the required FIRMS data

The FIRMS Area API accepts at most **5 days per request**.

Each date segment is therefore divided into chunks of at most five days.

Unlike the earlier version of this notebook, the loop explicitly advances `chunk_start` after each request.

In [ ]:
frames = []

for segment in download_segments:

    sensor_name = segment["sensor_name"]
    source = segment["source"]
    segment_start = segment["start"]
    segment_end = segment["end"]

    chunk_start = segment_start

    while chunk_start <= segment_end:

        remaining_days = (segment_end - chunk_start).days + 1
        chunk_days = min(5, remaining_days)

        url = (
            f"{BASE_AREA_URL}/{MAP_KEY}/{source}/"
            f"{bbox}/{chunk_days}/{chunk_start.isoformat()}"
        )

        print(
            f"Downloading {sensor_name:15s} | "
            f"{source:20s} | "
            f"{chunk_start} | {chunk_days} day(s)",
            end=" -> "
        )

        response = requests.get(url, timeout=120)
        response.raise_for_status()

        text = response.text.strip()

        # Check whether FIRMS returned a CSV table
        if "latitude" in text.lower():

            df_chunk = pd.read_csv(io.StringIO(text))

            print(f"{len(df_chunk)} detections")

            if not df_chunk.empty:

                df_chunk["sensor_name"] = sensor_name
                df_chunk["firms_source"] = source

                df_chunk["processing_type"] = (
                    "Standard Processing"
                    if source.endswith("_SP")
                    else "Near Real-Time"
                )

                frames.append(df_chunk)

        else:
            # Useful if FIRMS returns an error/message with HTTP 200
            print("non-data response")
            print("   FIRMS response:", text[:300])

        chunk_start += timedelta(days=chunk_days)


print()

if frames:
    print("Download complete.")
    print("Returned non-empty tables:", len(frames))
else:
    print(
        "Download complete, but FIRMS returned "
        "zero detections for this AOI and period."
    )

## 9. Combine all detections

All original FIRMS columns are retained.

Different products can have slightly different schemas, so `pandas.concat()` takes the union of all fields.

In [ ]:
df = pd.concat(frames, ignore_index=True, sort=False)

print(f"Downloaded detections before AOI clipping: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")

df.head()

## 10. Convert to points and clip to the true AOI

In [ ]:
fires = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df["longitude"],
        df["latitude"],
    ),
    crs="EPSG:4326",
)

fires = fires[fires.geometry.intersects(aoi_geom)].copy()

print(f"Detections inside AOI: {len(fires):,}")

## 11. Create a UTC acquisition timestamp

The original `acq_date` and `acq_time` attributes are kept.

A combined `acq_datetime` field is added for temporal mapping in QGIS.

In [ ]:
fires["acq_time_str"] = (
    fires["acq_time"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(4)
)

fires["acq_datetime"] = pd.to_datetime(
    fires["acq_date"].astype(str)
    + " "
    + fires["acq_time_str"].str[:2]
    + ":"
    + fires["acq_time_str"].str[2:],
    errors="coerce",
    utc=True,
)

fires = fires.sort_values("acq_datetime")

fires[
    [
        "sensor_name",
        "firms_source",
        "processing_type",
        "acq_date",
        "acq_time",
        "acq_datetime",
        "geometry",
    ]
].head()

## 12. Quick summary

This is useful for checking which FIRMS processing products actually contributed to the final dataset.

In [ ]:
summary = (
    fires.groupby(
        ["sensor_name", "firms_source", "processing_type"],
        dropna=False,
    )
    .size()
    .reset_index(name="detections")
)

summary

## 13. Export to GeoPackage

In [ ]:
fires.to_file(
    OUTPUT_FILE,
    layer=OUTPUT_LAYER,
    driver="GPKG",
)

print("Finished.")
print("Output file:", OUTPUT_FILE)
print("Layer:      ", OUTPUT_LAYER)
print(f"Points:      {len(fires):,}")
print("Period:     ", start_date, "to", end_date)

## Example configurations

### Near-real-time: last 7 days

```python
DATE_MODE = "recent"
DAYS = 7
```

### Historical: two weeks one year ago

```python
DATE_MODE = "range"
START_DATE = "2025-08-20"
END_DATE   = "2025-09-02"
```

### Historical fire event: exact period

```python
DATE_MODE = "range"
START_DATE = "2024-07-10"
END_DATE   = "2024-07-24"
```